# HISEMOTIONS — Baseline TF-IDF + Logistic Regression

Objetivo: obtener un micro F1 de referencia rápido sin GPU.

Pipeline:
1. TF-IDF con n-gramas (1,2) sobre texto limpio
2. Binary Relevance: un `LogisticRegression(class_weight='balanced')` por emoción
3. Búsqueda de umbral óptimo por label en dev

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, classification_report
import re
import warnings
warnings.filterwarnings('ignore')

EMOTION_COLS = ['anger', 'fear', 'joy', 'sadness', 'surprise', 'hope']

In [ ]:
def load_split(path):
    """Carga CSV, elimina filas con texto vacío y rellena labels NaN con 0."""
    df = pd.read_csv(path)
    df = df.dropna(subset=['text']).reset_index(drop=True)
    for col in EMOTION_COLS:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(int)
    return df

train_df = load_split('../train/train.csv')
dev_df   = load_split('../dev/dev.csv')
print(f'Train: {len(train_df)} | Dev: {len(dev_df)}')

## 1. Preprocesamiento básico del español histórico

In [ ]:
ABBREV = {
    r'\bv\.?m\.?d?\.?\b': 'vuestra merced',
    r'\bvmd\b': 'vuestra merced',
    r'\bvm\b': 'vuestra merced',
    r'\bv\.s\.\b': 'vuestra señoria',
    r'\bans[íi]\b': 'asi',
    r'\baquesta\b': 'esta',
    r'\baqueste\b': 'este',
    r'\bdeste\b': 'de este',
    r'\bdesta\b': 'de esta',
    r'\bdellos\b': 'de ellos',
    r'\bdellas\b': 'de ellas',
}

def clean_text(text: str) -> str:
    text = str(text).lower()
    for pattern, replacement in ABBREV.items():
        text = re.sub(pattern, replacement, text)
    text = re.sub(r'[^a-záéíóúüñ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['clean'] = train_df['text'].map(clean_text)
dev_df['clean']   = dev_df['text'].map(clean_text)

print('Ejemplo limpieza:')
print('Original: ', train_df['text'].iloc[0][:150])
print('Limpio:   ', train_df['clean'].iloc[0][:150])

## 2. TF-IDF vectorization

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50_000,
    sublinear_tf=True,
    min_df=2,
)

X_train = vectorizer.fit_transform(train_df['clean'])
X_dev   = vectorizer.transform(dev_df['clean'])
print(f'Feature matrix: {X_train.shape}')

## 3. Entrenamiento — Binary Relevance con Logistic Regression

In [ ]:
models = {}
dev_probs = np.zeros((len(dev_df), len(EMOTION_COLS)))

for i, emo in enumerate(EMOTION_COLS):
    y_train = train_df[emo].values
    clf = LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0)
    clf.fit(X_train, y_train)
    models[emo] = clf
    dev_probs[:, i] = clf.predict_proba(X_dev)[:, 1]

print('Modelos entrenados.')

## 4. Búsqueda de umbral óptimo por label (micro F1)

In [ ]:
y_dev = dev_df[EMOTION_COLS].values
thresholds = np.full(len(EMOTION_COLS), 0.5)

for i, emo in enumerate(EMOTION_COLS):
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.05, 0.96, 0.05):
        preds = (dev_probs >= thresholds).astype(int)
        preds[:, i] = (dev_probs[:, i] >= t).astype(int)
        f1 = f1_score(y_dev, preds, average='micro', zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    thresholds[i] = best_t

y_pred = (dev_probs >= thresholds).astype(int)
final_f1 = f1_score(y_dev, y_pred, average='micro', zero_division=0)

print('Umbrales óptimos:')
for emo, t in zip(EMOTION_COLS, thresholds):
    print(f'  {emo:10s}: {t:.2f}')
print(f'\nMicro F1 en dev (con umbral 0.5): {f1_score(y_dev, (dev_probs >= 0.5).astype(int), average="micro", zero_division=0):.4f}')
print(f'Micro F1 en dev (umbral óptimo):  {final_f1:.4f}')

## 5. Reporte por emoción

In [ ]:
print(classification_report(y_dev, y_pred, target_names=EMOTION_COLS, zero_division=0))

## 6. Alternativa: LinearSVC (suele ser más rápido y preciso con TF-IDF)

In [ ]:
svm_probs = np.zeros((len(dev_df), len(EMOTION_COLS)))

for i, emo in enumerate(EMOTION_COLS):
    y_train = train_df[emo].values
    # CalibratedClassifierCV necesario para obtener probabilidades con LinearSVC
    svc = CalibratedClassifierCV(
        LinearSVC(class_weight='balanced', max_iter=2000, C=0.5)
    )
    svc.fit(X_train, y_train)
    svm_probs[:, i] = svc.predict_proba(X_dev)[:, 1]

svm_pred = (svm_probs >= 0.5).astype(int)
svm_f1 = f1_score(y_dev, svm_pred, average='micro', zero_division=0)
print(f'LinearSVC micro F1 (umbral 0.5): {svm_f1:.4f}')

## 7. Guardar predicciones de dev para comparar con BERT

In [ ]:
import os, pickle
os.makedirs('../models', exist_ok=True)

# Guardar modelo y umbrales
with open('../models/tfidf_baseline.pkl', 'wb') as f:
    pickle.dump({'vectorizer': vectorizer, 'models': models, 'thresholds': thresholds}, f)

# Guardar predicciones en dev
dev_preds_df = dev_df[['text']].copy()
for i, emo in enumerate(EMOTION_COLS):
    dev_preds_df[emo] = y_pred[:, i]
dev_preds_df.to_csv('../submissions/baseline_dev_preds.csv', index=False)
print('Guardado.')